In [1]:
import json
from pathlib import Path

import numpy as np
from PIL import Image
from sahi.slicing import slice_coco
from skmultilearn.model_selection import iterative_train_test_split
from tqdm import tqdm

In [2]:
MSGO_CLASSES = {
    "Plane": 0,
    "Bridge": 1,
    "Airport": 2,
    "Harbor": 3,
    "Vehicle": 4,
    "Ship": 5,
}

DOTAV2_CLASSES = {
    "plane": 0,
    "bridge": 1,
    "airport": 2,
    "harbror": 3,
    "large-vehicle": 4,
    "small-vehicle": 4,
    "ship": 5,
}

NUM_CLASSES = len(MSGO_CLASSES)
CLASS_NAMES = list(MSGO_CLASSES.keys())
MSGO_CLASSES_REVERSED = {v: k for k, v in MSGO_CLASSES.items()}

In [11]:
Image.MAX_IMAGE_PIXELS = None


def dotav2_label_to_hbb(line: str, img_width: int, img_height: int) -> str:
    # 506.0 201.0 481.0 199.0 477.0 83.0 501.0 83.0 large-vehicle 0
    # class x_center y_center width height
    parts = line.strip().split()

    class_name = parts[-2]
    class_index = DOTAV2_CLASSES.get(class_name, -1)

    if class_index == -1:
        return ""

    poly = np.array(parts[:-2], dtype=np.float32).reshape(4, 2)

    xmin = np.min(poly[:, 0])
    xmax = np.max(poly[:, 0])
    ymin = np.min(poly[:, 1])
    ymax = np.max(poly[:, 1])

    x_center = (xmin + xmax) / 2 / img_width
    y_center = (ymin + ymax) / 2 / img_height
    width = (xmax - xmin) / img_width
    height = (ymax - ymin) / img_height

    return f"{class_index} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"


def dior_label_to_hbb(line: str) -> str:
    # 5 0.0775 0.32 0.0225 0.0125
    # class x_center y_center width height
    parts = line.strip().split()

    class_index = int(parts[0])

    mapping = {8: 0, 9: 1, 5: 4, 7: 5, 18: 2, 6: 3}
    if class_index not in mapping:
        return ""

    new_class_index = mapping[class_index]

    coords = list(map(float, parts[1:]))

    return f"{new_class_index} " + " ".join(f"{c:.6f}" for c in coords)


def convert_dotav2(root_dir: str) -> None:
    dota_path = Path(root_dir)
    images_path = dota_path / "images"
    labels_path = dota_path / "labels"

    for split_path in labels_path.iterdir():
        label_files = list(split_path.glob("*.txt"))
        for label_file in tqdm(label_files, desc=f"Converting DOTAv2 labels in split {split_path.stem}..."):
            img_file = images_path / split_path.stem / f"{label_file.stem}.jpg"

            with Image.open(img_file) as img:
                w, h = img.size

            with open(label_file) as f:
                lines = f.readlines()

            new_label_content = []
            for line in lines:
                hbb_line = dotav2_label_to_hbb(line, w, h)
                if hbb_line:
                    new_label_content.append(hbb_line)

            with open(label_file, "w") as f:
                f.write("\n".join(new_label_content))


def convert_dior(root_dir: str) -> None:
    dior_path = Path(root_dir)

    labels_dir = dior_path / "labels"

    label_files = list(labels_dir.glob("*.txt"))
    for label_file in tqdm(label_files, desc="Converting DIOR labels..."):
        with open(label_file) as f:
            lines = f.readlines()

        new_label_content = []
        for line in lines:
            hbb_line = dior_label_to_hbb(line)
            if hbb_line:
                new_label_content.append(hbb_line)

        with open(label_file, "w") as f:
            f.write("\n".join(new_label_content))


convert_dotav2("D:\\stuff\\datasets\\MSGOv1\\DOTAv2")
convert_dior("D:\\stuff\\datasets\\MSGOv1\\DIOR")

Converting DOTAv2 labels in split val...: 100%|██████████| 593/593 [00:29<00:00, 20.15it/s]


In [12]:
def restructure_dotav2_folder(dota_dir: str):
    dota_path = Path(dota_dir)
    train_images_path = dota_path / "images" / "train"
    val_images_path = dota_path / "images" / "val"
    train_labels_path = dota_path / "labels" / "train"
    val_labels_path = dota_path / "labels" / "val"
    new_images_path = dota_path.parent / "combined" / "images"
    new_labels_path = dota_path.parent / "combined" / "labels"

    new_images_path.mkdir(parents=True, exist_ok=True)
    new_labels_path.mkdir(parents=True, exist_ok=True)

    train_images = list(train_images_path.glob("*.jpg"))
    for img_file in tqdm(train_images, desc="Moving DOTA train images"):
        img_file.rename(new_images_path / img_file.name)

    val_images = list(val_images_path.glob("*.jpg"))
    for img_file in tqdm(val_images, desc="Moving DOTA val images"):
        img_file.rename(new_images_path / img_file.name)

    train_labels = list(train_labels_path.glob("*.txt"))
    for label_file in tqdm(train_labels, desc="Moving DOTA train labels"):
        label_file.rename(new_labels_path / label_file.name)

    val_labels = list(val_labels_path.glob("*.txt"))
    for label_file in tqdm(val_labels, desc="Moving DOTA val labels"):
        label_file.rename(new_labels_path / label_file.name)

    for root, dirs, _ in dota_path.walk(top_down=False):
        for name in dirs:
            (root / name).rmdir()

    dota_path.rmdir()


def restructure_dior_folder(dior_path: str):
    dior_path = Path(dior_path)
    images_path = dior_path / "images"
    labels_path = dior_path / "labels"
    new_images_path = dior_path.parent / "combined" / "images"
    new_labels_path = dior_path.parent / "combined" / "labels"

    images = list(images_path.glob("*.jpg"))
    for img_file in tqdm(images, desc="Moving DIOR images"):
        img_file.rename(new_images_path / img_file.name)

    labels = list(labels_path.glob("*.txt"))
    for label_file in tqdm(labels, desc="Moving DIOR labels"):
        label_file.rename(new_labels_path / label_file.name)

    for root, dirs, _ in dior_path.walk(top_down=False):
        for name in dirs:
            (root / name).rmdir()

    dior_path.rmdir()


restructure_dotav2_folder("D:\\stuff\\datasets\\MSGOv1\\DOTAv2")
restructure_dior_folder("D:\\stuff\\datasets\\MSGOv1\\DIOR")

Moving DIOR labels: 100%|██████████| 23463/23463 [00:08<00:00, 2630.70it/s]

images
labels


In [14]:
def get_empty_image_paths(root_path: Path) -> list[Path]:
    results = []

    images_dir = root_path / "images"
    labels_dir = root_path / "labels"

    for img_file in images_dir.iterdir():
        img_name = img_file.stem
        label_file = labels_dir / f"{img_name}.txt"

        if label_file.stat().st_size == 0:
            results.append(img_file)

    return results


def delete_empty_images(root_dir: str) -> None:
    root_path = Path(root_dir)

    labels_dir = root_path / "labels"

    empty_image_paths = get_empty_image_paths(root_path)

    for img_file in tqdm(empty_image_paths, desc="Deleting empty images and labels..."):
        label_file = labels_dir / f"{img_file.stem}.txt"

        try:
            img_file.unlink()
        except Exception as e:
            print(f"Failed to delete image {img_file}: {e}")

        if label_file.exists():
            try:
                label_file.unlink()
            except Exception as e:
                print(f"Failed to delete label {label_file}: {e}")


delete_empty_images("D:\\stuff\\datasets\\MSGOv1\\combined")

Deleting empty images and labels...: 100%|██████████| 11463/11463 [00:08<00:00, 1336.75it/s]


In [16]:
def build_image_to_counts(root_dir: str) -> dict[str, dict[int, int]]:
    root_path = Path(root_dir)
    image_to_counts = {}

    images_dir = root_path / "images"
    labels_dir = root_path / "labels"

    label_files = list(labels_dir.glob("*.txt"))
    for label_file in tqdm(label_files, desc="Counting"):
        img_file = images_dir / f"{label_file.stem}.jpg"

        counts = {}
        with open(label_file) as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            counts[class_id] = counts.get(class_id, 0) + 1

        image_to_counts[str(img_file)] = counts

    return image_to_counts


image_to_counts = build_image_to_counts("D:\\stuff\\datasets\\MSGOv1\\combined")

Counting: 100%|██████████| 14423/14423 [02:28<00:00, 97.14it/s] 


In [17]:
image_paths = list(image_to_counts.keys())
num_images = len(image_paths)

y_counts = np.zeros((num_images, NUM_CLASSES), dtype=int)
for i, path in enumerate(image_paths):
    counts = image_to_counts[path]
    for class_id, count in counts.items():
        y_counts[i, class_id] = count

X = np.array(image_paths).reshape(-1, 1)

In [78]:
X_train, y_train_counts, X_temp, y_temp_counts = iterative_train_test_split(X, y_counts, test_size=0.2)
X_val, y_val_counts, X_test, y_test_counts = iterative_train_test_split(X_temp, y_temp_counts, test_size=0.5)

X_train_paths = X_train.flatten().tolist()
X_val_paths = X_val.flatten().tolist()
X_test_paths = X_test.flatten().tolist()

print(f"Total images: {len(X)}")
print(f"Train images: {len(X_train_paths)} ({len(X_train_paths) / len(X):.1%})")
print(f"Validation images: {len(X_val_paths)} ({len(X_val_paths) / len(X):.1%})")
print(f"Test images: {len(X_test_paths)} ({len(X_test_paths) / len(X):.1%})")

Total images: 14423
Train images: 11551 (80.1%)
Validation images: 1439 (10.0%)
Test images: 1433 (9.9%)


In [80]:
def check_distribution(paths, image_to_counts_map, num_classes):
    total_counts = np.zeros(num_classes, dtype=int)
    for path in paths:
        counts = image_to_counts_map.get(path, {})
        for class_id, count in counts.items():
            total_counts[class_id] += count
    return total_counts


train_counts = check_distribution(X_train_paths, image_to_counts, NUM_CLASSES)
val_counts = check_distribution(X_val_paths, image_to_counts, NUM_CLASSES)
test_counts = check_distribution(X_test_paths, image_to_counts, NUM_CLASSES)
total_counts = train_counts + val_counts + test_counts

print(f"Class Names: {list(MSGO_CLASSES.keys())}")
print(f"Total Instances: {total_counts}")
print(f"Train Instances: {train_counts} ({(train_counts / total_counts * 100).round(1)}%)")
print(f"Val Instances:   {val_counts} ({(val_counts / total_counts * 100).round(1)}%)")
print(f"Test Instances:  {test_counts} ({(test_counts / total_counts * 100).round(1)}%)")

Class Names: ['Plane', 'Bridge', 'Airport', 'Harbor', 'Vehicle', 'Ship']
Total Instances: [ 21469   7010   1737   5509 289641 116753]
Train Instances: [ 20065   6215   1390   4505 280450 104371] ([93.5 88.7 80.  81.8 96.8 89.4]%)
Val Instances:   [ 717  397  172  482 4723 6431] ([3.3 5.7 9.9 8.7 1.6 5.5]%)
Test Instances:  [ 687  398  175  522 4468 5951] ([ 3.2  5.7 10.1  9.5  1.5  5.1]%)


In [51]:
def yolo_hbb_to_coco(
    yolo_coords_norm: list[float], img_width: int, img_height: int
) -> tuple[list[float], list[float], float]:
    x_center_norm, y_center_norm, width_norm, height_norm = yolo_coords_norm

    width_abs = width_norm * img_width
    height_abs = height_norm * img_height
    x_center_abs = x_center_norm * img_width
    y_center_abs = y_center_norm * img_height

    x_min = x_center_abs - (width_abs / 2)
    y_min = y_center_abs - (height_abs / 2)

    coco_bbox = [
        round(x_min, 2),
        round(y_min, 2),
        round(width_abs, 2),
        round(height_abs, 2),
    ]

    x_max, y_max = x_min + width_abs, y_min + height_abs
    coco_segmentation = [
        round(x_min, 2),
        round(y_min, 2),
        round(x_max, 2),
        round(y_min, 2),
        round(x_max, 2),
        round(y_max, 2),
        round(x_min, 2),
        round(y_max, 2),
    ]

    area = width_abs * height_abs

    return coco_segmentation, coco_bbox, round(area, 2)


def create_coco_json_from_hbb(root_dir: str, json_name: str, show_bad_annotations: bool = False) -> None:
    coco_data = {
        "info": {"description": "Pre-sliced MSGO HBB dataset"},
        "licenses": [],
        "categories": [
            {"id": cid, "name": cname, "supercategory": "object"} for cid, cname in MSGO_CLASSES_REVERSED.items()
        ],
        "images": [],
        "annotations": [],
    }

    root_path = Path(root_dir)
    image_id_counter, annotation_id_counter = 1, 1
    skipped_annotations_count = 0

    images_dir = root_path / "images"
    labels_dir = root_path / "labels"

    label_files = list(labels_dir.glob("*.txt"))

    for label_file in tqdm(label_files, desc="Processing"):
        img_file = images_dir / f"{label_file.stem}.jpg"

        with Image.open(img_file) as img:
            img_width, img_height = img.size

        image_info = {
            "id": image_id_counter,
            "file_name": img_file.relative_to(root_path).as_posix(),
            "width": img_width,
            "height": img_height,
        }
        coco_data["images"].append(image_info)

        with open(label_file) as f:
            lines = f.readlines()

        for line_num, line in enumerate(lines, 1):
            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            yolo_obb_data = [float(p) for p in parts[1:]]

            segmentation, bbox, area = yolo_hbb_to_coco(yolo_obb_data, img_width, img_height)

            reason = ""
            if bbox[2] <= 0 or bbox[3] <= 0 or area <= 1:
                skipped_annotations_count += 1
                if show_bad_annotations:
                    print(f"\nFile: {label_file.name}")
                    print(f"Line in File: {line_num}")
                    print(f"Reason: {reason}")
                    print(f"Bbox [x,y,w,h]: {bbox}")
                    print(f"Area: {area}")
                    print(f"Segmentation: {segmentation}")
                continue

            annotation_info = {
                "id": annotation_id_counter,
                "image_id": image_id_counter,
                "category_id": class_id,
                "bbox": bbox,
                "segmentation": [segmentation],
                "area": area,
                "iscrowd": 0,
            }
            coco_data["annotations"].append(annotation_info)
            annotation_id_counter += 1

        image_id_counter += 1

    print(f"\nProcessed {image_id_counter - 1} images and {annotation_id_counter - 1} annotations.")
    print(f"Skipped {skipped_annotations_count} annotations.")

    ouput_json_path = root_path / json_name
    with open(ouput_json_path, "w") as f:
        json.dump(coco_data, f)


create_coco_json_from_hbb("D:\\stuff\\datasets\\MSGOv1\\combined", "master_annotations.coco.json", False)

Processing: 100%|██████████| 14423/14423 [10:15<00:00, 23.43it/s]



Processed 14423 images and 442112 annotations.
Skipped 7 annotations.


In [52]:
ROOT_DIR = Path("D:\\stuff\\datasets\\MSGOv1\\combined")
MASTER_COCO_PATH = ROOT_DIR / "master_annotations.coco.json"
FINAL_DATASET_DIR = ROOT_DIR.parent / "sliced"


def slice_split(image_paths, master_coco_data, split_name):
    output_dir = FINAL_DATASET_DIR / split_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing {split_name} split...")

    split_relative_paths = {Path(p).relative_to(ROOT_DIR).as_posix() for p in image_paths}
    split_images = [img for img in master_coco_data["images"] if img["file_name"] in split_relative_paths]
    split_image_ids = {img["id"] for img in split_images}
    split_annotations = [ann for ann in master_coco_data["annotations"] if ann["image_id"] in split_image_ids]

    subset_coco_data = {
        "images": split_images,
        "annotations": split_annotations,
        "categories": master_coco_data["categories"],
    }

    subset_coco_path = output_dir / f"{split_name}_subset.json"
    with open(subset_coco_path, "w") as f:
        json.dump(subset_coco_data, f)

    slice_coco(
        coco_annotation_file_path=subset_coco_path,
        image_dir=ROOT_DIR,
        output_dir=output_dir,
        output_coco_annotation_file_name="_annotations",
        slice_height=800,
        slice_width=800,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        min_area_ratio=0.4,
        ignore_negative_samples=False,
        verbose=False,
    )

    subset_coco_path.unlink()

In [53]:
with open(MASTER_COCO_PATH) as f:
    master_data = json.load(f)

slice_split(X_test_paths, master_data, "test")
slice_split(X_val_paths, master_data, "valid")
slice_split(X_train_paths, master_data, "train")


Processing test split...


100%|██████████| 2162/2162 [02:18<00:00, 15.61it/s]



Processing valid split...


100%|██████████| 2156/2156 [03:04<00:00, 11.65it/s]



Processing train split...


100%|██████████| 10105/10105 [49:18<00:00,  3.42it/s] 


In [54]:
def move_png_files(source_folder: str) -> None:
    source_path = Path(source_folder)
    dest_path = source_path / "images"
    dest_path.mkdir(parents=True, exist_ok=True)

    png_files = list(source_path.glob("*.png"))
    for png_file in tqdm(png_files, desc="Moving files..."):
        png_file.rename(dest_path / png_file.name)


move_png_files("D:\\stuff\\datasets\\MSGOv1\\sliced\\test")
move_png_files("D:\\stuff\\datasets\\MSGOv1\\sliced\\train")
move_png_files("D:\\stuff\\datasets\\MSGOv1\\sliced\\valid")

Moving files...: 100%|██████████| 3064/3064 [00:02<00:00, 1455.05it/s]


In [55]:
def rename_json(json: str):
    json_path = Path(json)
    json_path.rename(json_path.parent / "_annotations.coco.json")


rename_json("D:\\stuff\\datasets\\MSGOv1\\sliced\\test\\_annotations_coco.json")
rename_json("D:\\stuff\\datasets\\MSGOv1\\sliced\\train\\_annotations_coco.json")
rename_json("D:\\stuff\\datasets\\MSGOv1\\sliced\\valid\\_annotations_coco.json")

In [56]:
from collections import defaultdict


def coco_to_yolo_hbb(split_path: Path):
    json_path = split_path / "_annotations.coco.json"
    labels_path = split_path / "labels"

    labels_path.mkdir(parents=True, exist_ok=True)
    with open(json_path) as f:
        coco_data = json.load(f)

    annotations_by_image_id = defaultdict(list)
    for ann in coco_data["annotations"]:
        annotations_by_image_id[ann["image_id"]].append(ann)

    for image_info in tqdm(coco_data["images"], desc=f"Creating YOLO HBB files for '{split_path.name}'"):
        image_id = image_info["id"]
        img_width = image_info["width"]
        img_height = image_info["height"]

        label_filename = Path(image_info["file_name"]).stem + ".txt"
        output_path = labels_path / label_filename

        yolo_lines = []
        if image_id in annotations_by_image_id:
            for ann in annotations_by_image_id[image_id]:
                class_id = ann["category_id"]

                x_min, y_min, width, height = ann["bbox"]

                x_center_norm = (x_min + width / 2) / img_width
                y_center_norm = (y_min + height / 2) / img_height
                width_norm = width / img_width
                height_norm = height / img_height

                yolo_lines.append(
                    f"{class_id} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f}"
                )

        with open(output_path, "w") as f:
            f.write("\n".join(yolo_lines))


def create_label_files_from_master_json(root_dir: str):
    sliced_root_path = Path(root_dir)

    splits_to_process = [d for d in sliced_root_path.iterdir() if d.is_dir()]

    for split_path in splits_to_process:
        print(f"Processing split: {split_path.name}")
        coco_to_yolo_hbb(split_path)


create_label_files_from_master_json("D:\\stuff\\datasets\\MSGOv1\\sliced")

Processing split: test


Creating YOLO HBB files for 'test': 100%|██████████| 3032/3032 [00:01<00:00, 2228.32it/s]


Processing split: train


Creating YOLO HBB files for 'train': 100%|██████████| 53543/53543 [00:28<00:00, 1902.00it/s]


Processing split: valid


Creating YOLO HBB files for 'valid': 100%|██████████| 3064/3064 [00:01<00:00, 1544.28it/s]


In [57]:
def empty_image_percentages(root_dir):
    root_path = Path(root_dir)
    results = {}
    total_empty = 0
    total_images = 0

    for split_path in root_path.iterdir():
        images_dir = split_path / "images"
        labels_dir = split_path / "labels"

        split_total = 0
        split_empty = 0

        for img_file in images_dir.iterdir():
            split_total += 1
            img_name = img_file.stem
            label_file = labels_dir / f"{img_name}.txt"
            if label_file.exists() and label_file.stat().st_size == 0:
                split_empty += 1

        results[split_path.name] = {
            "empty_count": split_empty,
            "total": split_total,
            "percentage": round((split_empty / split_total) * 100, 2),
        }

        total_empty += split_empty
        total_images += split_total

    overall_percentage = round((total_empty / total_images) * 100, 2)

    for split, stats in results.items():
        print(f"{split}: {stats['empty_count']} / {stats['total']} ({stats['percentage']}%)")

    print(f"\nOverall: {total_empty} / {total_images} ({overall_percentage}%)")


empty_image_percentages("D:\\stuff\\datasets\\MSGOv1\\sliced")

test: 340 / 3032 (11.21%)
train: 24479 / 53543 (45.72%)
valid: 359 / 3064 (11.72%)

Overall: 25178 / 59639 (42.22%)


In [59]:
import random


def delete_some_empty_images(root_dir: str, empty_target_ratio: float = 0.05) -> None:
    root_path = Path(root_dir)

    for split_path in root_path.iterdir():
        if not split_path.is_dir():
            continue

        images_dir = split_path / "images"
        labels_dir = split_path / "labels"

        all_images = list(images_dir.iterdir())
        empty_images = get_empty_image_paths(split_path)

        num_total = len(all_images)
        num_empty = len(empty_images)
        target_empty = round(num_total * empty_target_ratio)

        if num_empty <= target_empty:
            print(f"[{split_path.name}] {num_empty}/{num_total} empty images already <= target {target_empty}")
            continue

        num_to_delete = num_empty - target_empty
        images_to_delete = random.sample(empty_images, num_to_delete)

        for img_file in tqdm(images_to_delete, desc=f"Deleting empty images in {split_path.name}"):
            label_file = labels_dir / f"{img_file.stem}.txt"
            try:
                img_file.unlink()
                label_file.unlink()
            except Exception as e:
                print(f"Failed to delete {img_file}: {e}")


delete_some_empty_images("D:\\stuff\\datasets\\MSGOv1\\sliced")
empty_image_percentages("D:\\stuff\\datasets\\MSGOv1\\sliced")

Deleting empty images in valid: 100%|██████████| 10/10 [00:00<00:00, 2376.65it/s]


test: 142 / 2834 (5.01%)
train: 1587 / 30651 (5.18%)
valid: 143 / 2848 (5.02%)

Overall: 1872 / 36333 (5.15%)


In [60]:
def to_jpg(image_path: Path) -> None:
    file_name = image_path.stem + ".jpg"
    jpg_path = image_path.parent / file_name

    img = Image.open(image_path)
    rgb_img = img.convert("RGB")
    rgb_img.save(jpg_path)


def convert_images(root_dir: str) -> None:
    root_path = Path(root_dir)

    for split_path in root_path.iterdir():
        images_dir = split_path / "images"

        img_files = list(images_dir.glob("*.png"))
        for img_file in tqdm(img_files, desc=f"Converting images in {split_path.stem}"):
            if img_file.suffix == ".png":
                to_jpg(img_file)

                try:
                    img_file.unlink()
                except Exception as e:
                    print(f"Failed to delete image {img_file}: {e}")


convert_images("D:\\stuff\\datasets\\MSGOv1\\sliced")

Converting images in valid: 100%|██████████| 2848/2848 [04:14<00:00, 11.20it/s]


In [61]:
def delete_json(json: str):
    json_path = Path(json)
    json_path.unlink()


delete_json("D:\\stuff\\datasets\\MSGOv1\\sliced\\test\\_annotations.coco.json")
delete_json("D:\\stuff\\datasets\\MSGOv1\\sliced\\train\\_annotations.coco.json")
delete_json("D:\\stuff\\datasets\\MSGOv1\\sliced\\valid\\_annotations.coco.json")

In [62]:
create_coco_json_from_hbb("D:\\stuff\\datasets\\MSGOv1\\sliced\\test", "_annotations.coco.json", False)
create_coco_json_from_hbb("D:\\stuff\\datasets\\MSGOv1\\sliced\\train", "_annotations.coco.json", False)
create_coco_json_from_hbb("D:\\stuff\\datasets\\MSGOv1\\sliced\\valid", "_annotations.coco.json", False)

Processing:   0%|          | 0/2834 [00:00<?, ?it/s]

Processing: 100%|██████████| 2834/2834 [00:47<00:00, 59.40it/s]



Processed 2834 images and 37504 annotations.
Skipped 0 annotations.


Processing: 100%|██████████| 30651/30651 [09:09<00:00, 55.82it/s] 



Processed 30651 images and 633567 annotations.
Skipped 0 annotations.


Processing: 100%|██████████| 2848/2848 [00:48<00:00, 58.17it/s]



Processed 2848 images and 43198 annotations.
Skipped 0 annotations.


In [63]:
msgo_path = Path("D:\\stuff\\datasets\\MSGOv1\\sliced")
msgo_path.rename(msgo_path.parent / "MSGOv1")

WindowsPath('D:/stuff/datasets/MSGOv1/MSGOv1')

In [65]:
def verify_final_distribution(root_dir: str):
    root_path = Path(root_dir)
    all_split_data = {}
    splits_to_process = [d for d in root_path.iterdir() if d.is_dir()]

    for split_path in splits_to_process:
        split_name = split_path.name
        labels_dir = split_path / "labels"

        split_counts = np.zeros(NUM_CLASSES, dtype=int)
        annotated_file_count = 0
        empty_file_count = 0

        label_files = list(labels_dir.glob("*.txt"))

        for label_file in tqdm(label_files, desc=f"Analyzing '{split_name}' labels"):
            with open(label_file) as f:
                lines = f.readlines()

            if not lines:
                empty_file_count += 1
            else:
                annotated_file_count += 1
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        if 0 <= class_id < NUM_CLASSES:
                            split_counts[class_id] += 1

        all_split_data[split_name] = {
            "counts": split_counts,
            "annotated_files": annotated_file_count,
            "empty_files": empty_file_count,
        }

    total_counts = np.zeros(NUM_CLASSES, dtype=int)
    for data in all_split_data.values():
        total_counts += data["counts"]

    print(f"Class Names: {CLASS_NAMES}")
    print(f"Total Instances: {total_counts}")

    for split_name, data in all_split_data.items():
        split_counts = data["counts"]
        percentages = np.round((split_counts / (total_counts + 1e-9)) * 100, 1)
        print(f"{split_name.capitalize():<6} Instances: {split_counts} ({percentages}%)")

    for split_name, data in all_split_data.items():
        annotated = data["annotated_files"]
        empty = data["empty_files"]
        total = annotated + empty
        print(
            f"{split_name.capitalize():<6}: {annotated} images with annotations, {empty} empty images (Total: {total})"
        )


verify_final_distribution("D:\\stuff\\datasets\\MSGOv1\\MSGOv1")

Analyzing 'valid' labels: 100%|██████████| 2848/2848 [00:00<00:00, 6348.23it/s]

Class Names: ['Plane', 'Bridge', 'Airport', 'Harbor', 'Vehicle', 'Ship']
Total Instances: [ 31653   8816   2934   5520 497398 167948]
Test   Instances: [ 1482   827   261   788 13515 20631] ([ 4.7  9.4  8.9 14.3  2.7 12.3]%)
Train  Instances: [ 28736   7169   2412   4005 469355 121890] ([90.8 81.3 82.2 72.6 94.4 72.6]%)
Valid  Instances: [ 1435   820   261   727 14528 25427] ([ 4.5  9.3  8.9 13.2  2.9 15.1]%)
Test  : 2692 images with annotations, 142 empty images (Total: 2834)
Train : 29064 images with annotations, 1587 empty images (Total: 30651)
Valid : 2705 images with annotations, 143 empty images (Total: 2848)
